# Web Scraping

# Tarefa 1 - Wikipedia (Steve Jobs)

In [15]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import unquote
import os, time, csv
from datetime import datetime

In [ ]:
url_inicial = "https://pt.wikipedia.org/wiki/Steve_Jobs"

requisicao = requests.get(url_inicial, headers={'User-Agent': 'aula-cpa'})
print(requisicao)

soup = BeautifulSoup(requisicao.text, "html.parser")

In [ ]:
titulo_pagina = soup.head.title.string #titulo da pg
print("Título da página: ", titulo_pagina)

primeiro_paragrafo = soup.find('p') #primeiro paragrafo da pg
print("Primeiro parágrafo: ", primeiro_paragrafo.text)

0 links internos encontrados


In [ ]:
links_internos = [] #links internos /wiki/
for a in soup.select("a[href^='/wiki/']"):
    link = "https://pt.wikipedia.org" + a["href"]
    links_internos.append(link)

print("Número de links internos: ", len(links_internos))
print(links_internos[:5])

In [ ]:
links_imagens = [] #src das imagens
for img in soup.select("img"):
    if img.has_attr("src"):
        links_imagens.append(img["src"])

print("Número de imagens encontradas: ", len(links_imagens))

In [ ]:
links_corrigidos_imagens = [] #links corrigidos para imagens
for link in links_imagens:
    if link.startswith("//"):
        link = "https:" + link
    elif link.startswith("/"):
        link = "https://pt.wikipedia.org" + link
    links_corrigidos_imagens.append(link)

print("Links corrigidos: ", len(links_corrigidos_imagens))
print(links_corrigidos_imagens[:5])

In [ ]:
def salvar_html(url, pasta="htmls_lab1"): #pasta onde os arquivos serao salvos
    os.makedirs(pasta, exist_ok=True)
    try:
        resposta = requests.get(url, headers={'User-Agent': 'aula-cpa'})
        resposta.raise_for_status()

        #acentos e caracteres especiais no nome do arquivo
        nome_arquivo = unquote(url.split("/")[-1]) + ".html" 
        caminho = os.path.join(pasta, nome_arquivo)

        with open(caminho, "w", encoding="utf-8") as arquivo:
            arquivo.write(resposta.text)
        return caminho

    except Exception as e:
        print("Erro ao baixar: ", url, "->", e)
        return None

salvar_html(url_inicial)
print("Pagina inicial salva")

In [ ]:
paginas = {url_inicial: requisicao.text} 

for i, url in enumerate(links_internos): 
    try:
        resp = requests.get(url, headers={'User-Agent': 'aula-cpa'}, timeout=10)
        paginas[url] = resp.text
        salvar_html(url)
    except Exception as e:
        print(f"Erro em {url}: {e}")

    if (i + 1) % 20 == 0:
        print(f"{i + 1}/{len(links_internos)} páginas baixadas...")

    time.sleep(0.5)

print(f"Total de paginas baixadas: {len(paginas)}")

In [ ]:
#extrair os dados de cada pg
dados_paginas = []
dados_imagens = []

for url, html in paginas.items():
    s = BeautifulSoup(html, "html.parser")
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    titulo = s.head.title.string if s.head and s.head.title else "N/A"

    p = s.find('p')
    primeiro_paragrafo = p.text if p else "N/A"

    links_pag = ["https://pt.wikipedia.org" + a["href"] for a in s.select("a[href^='/wiki/']")]

    imgs = []
    for img in s.select("img"):
        if img.has_attr("src"):
            src = img["src"]
            if src.startswith("//"):
                src = "https:" + src
            elif src.startswith("/"):
                src = "https://pt.wikipedia.org" + src
            imgs.append(src)
            dados_imagens.append({"pagina_url": url, "titulo": titulo, "imagem_url": src, "timestamp": timestamp})

    dados_paginas.append({
        "url": url,
        "titulo": titulo,
        "primeiro_paragrafo": primeiro_paragrafo,
        "qtd_imagens": len(imgs),
        "links_internos": " | ".join(links_pag),
        "timestamp": timestamp
    })

print(f"Paginas processadas: {len(dados_paginas)}")
print(f"Imagens encontradas: {len(dados_imagens)}")